## 1. Initial Data Inspection

This section performs an initial exploration of the US Regional Sales dataset to understand its structure, size, column characteristics, and overall contents. It examines the dataset dimensions, data types, missing values, unique values, duplicate records, and sample rows to establish a baseline understanding of the data before proceeding with detailed data quality assessment and preparation.

In [1]:
import pandas as pd

path = '../data/raw/US_Regional_Sales_Data.csv'
df = pd.read_csv(path)

def section(title):
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")

# 1. Shape
section("SHAPE")
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")

# 2. Column summary: dtype, nulls, unique count
section("COLUMN SUMMARY")
summary = pd.DataFrame({
    'Dtype':    df.dtypes.astype(str),
    'Nulls':    df.isna().sum(),
    'Null %':   (df.isna().mean() * 100).round(2),
    'Unique':   df.nunique(),
})
print(summary.to_string())

# 3. Sample rows
section("SAMPLE ROWS (first 3)")
print(df.head(3).T.to_string(header=['Row 0', 'Row 1', 'Row 2']))


SHAPE
Rows:    7,991
Columns: 16
Exact duplicate rows: 0

COLUMN SUMMARY
                    Dtype  Nulls  Null %  Unique
OrderNumber        object      0     0.0    7991
Sales Channel      object      0     0.0       4
WarehouseCode      object      0     0.0       6
ProcuredDate       object      0     0.0      11
OrderDate          object      0     0.0     945
ShipDate           object      0     0.0     966
DeliveryDate       object      0     0.0     966
CurrencyCode       object      0     0.0       1
_SalesTeamID        int64      0     0.0      28
_CustomerID         int64      0     0.0      50
_StoreID            int64      0     0.0     367
_ProductID          int64      0     0.0      47
Order Quantity      int64      0     0.0       8
Discount Applied  float64      0     0.0       7
Unit Cost          object      0     0.0    5252
Unit Price         object      0     0.0     664

SAMPLE ROWS (first 3)
                         Row 0         Row 1         Row 2
OrderNumber

In [2]:
d = df.copy()

date_cols = ['ProcuredDate', 'OrderDate', 'ShipDate', 'DeliveryDate']
money_cols = ['Unit Cost', 'Unit Price']

# Parse dates (day-first) and money (strip thousands separators)
for c in date_cols:
    d[c + '_p'] = pd.to_datetime(d[c], format='%d-%m-%Y', errors='coerce')
for c in money_cols:
    d[c + '_p'] = pd.to_numeric(d[c].astype(str).str.replace(',', '', regex=False), errors='coerce')

def section(title):
    print(f"\n{'=' * 50}\n{title}\n{'=' * 50}")

# 1. Overview
section("DATASET OVERVIEW")
print(f"Rows:                {len(df):,}")
print(f"Unique order numbers: {df['OrderNumber'].nunique():,}")
print(f"Duplicate order nums: {df['OrderNumber'].duplicated().sum():,}")
print(f"Warehouses:           {df['WarehouseCode'].nunique()}")

# 2. Date ranges
section("DATE RANGES & PARSE FAILURES")
dates = pd.DataFrame({
    c: {
        'Min': d[c + '_p'].min().date() if d[c + '_p'].notna().any() else None,
        'Max': d[c + '_p'].max().date() if d[c + '_p'].notna().any() else None,
        'Unparsed': int(d[c + '_p'].isna().sum()),
    } for c in date_cols
}).T
print(dates.to_string())

# 3. Date logic checks
section("DATE LOGIC CHECKS (should all be 0)")
checks = pd.Series({
    'Procured after order':   (d['ProcuredDate_p'] > d['OrderDate_p']).sum(),
    'Order after ship':       (d['OrderDate_p'] > d['ShipDate_p']).sum(),
    'Ship after delivery':    (d['ShipDate_p'] > d['DeliveryDate_p']).sum(),
    'Delivery before order':  (d['DeliveryDate_p'] < d['OrderDate_p']).sum(),
}, name='Violations').astype(int)
print(checks.to_string())

# 4. Money
section("MONEY COLUMNS")
money = pd.DataFrame({
    c: {
        'Min': d[c + '_p'].min(),
        'Max': d[c + '_p'].max(),
        'Unparsed': int(d[c + '_p'].isna().sum()),
    } for c in money_cols
}).T
print(money.round(2).to_string())

# 5. Quantity & discount
section("QUANTITY & DISCOUNT RANGE")
print(f"Order Quantity:   {df['Order Quantity'].min()} to {df['Order Quantity'].max()}")
print(f"Discount Applied: {df['Discount Applied'].min():.2f} to {df['Discount Applied'].max():.2f}")

# 6. Categorical counts
for col, title in [('CurrencyCode', 'CURRENCY'), ('Sales Channel', 'SALES CHANNELS')]:
    section(title)
    vc = df[col].value_counts()
    print(pd.DataFrame({'Count': vc, '%': (vc / vc.sum() * 100).round(1)}).to_string())


DATASET OVERVIEW
Rows:                7,991
Unique order numbers: 7,991
Duplicate order nums: 0
Warehouses:           6

DATE RANGES & PARSE FAILURES
                     Min         Max Unparsed
ProcuredDate  2017-12-31  2020-09-26        0
OrderDate     2018-05-31  2020-12-30        0
ShipDate      2018-06-02  2021-01-24        0
DeliveryDate  2018-06-07  2021-02-02        0

DATE LOGIC CHECKS (should all be 0)
Procured after order     0
Order after ship         0
Ship after delivery      0
Delivery before order    0

MONEY COLUMNS
               Min      Max  Unparsed
Unit Cost    68.68  5498.56       0.0
Unit Price  167.50  6566.00       0.0

QUANTITY & DISCOUNT RANGE
Order Quantity:   1 to 8
Discount Applied: 0.05 to 0.40

CURRENCY
              Count      %
CurrencyCode              
USD            7991  100.0

SALES CHANNELS
               Count     %
Sales Channel             
In-Store        3298  41.3
Online          2425  30.3
Distributor     1375  17.2
Wholesale        893

## 2. Data Quality Assessment

This section investigates the dataset for missing values and examines the data types of individual columns. The objective is to identify potential data quality issues and determine which columns require further validation or transformation before analysis.

In [3]:
# Missing values by column
missing = pd.DataFrame({
    'Missing': df.isna().sum(),
    'Missing %': (df.isna().mean() * 100).round(2),
}).sort_values('Missing', ascending=False)

print(f"\n{'=' * 45}\nMISSING VALUES BY COLUMN\n{'=' * 45}")
print(missing.to_string())

# Final result
total_missing = int(missing['Missing'].sum())
cols_affected = int((missing['Missing'] > 0).sum())
total_cells = df.shape[0] * df.shape[1]

print(f"\n{'=' * 45}\nFINAL RESULT\n{'=' * 45}")
print(f"Total missing cells:  {total_missing:,} of {total_cells:,} ({total_missing / total_cells * 100:.2f}%)")
print(f"Columns affected:     {cols_affected} of {df.shape[1]}")
print(f"Rows with any missing: {int(df.isna().any(axis=1).sum()):,} of {df.shape[0]:,}")

if total_missing == 0:
    print("Verdict: Dataset is complete. No imputation or dropping needed.")
else:
    print(f"Verdict: Most affected column is '{missing.index[0]}' "
          f"({missing['Missing %'].iloc[0]}% missing). Decide whether to impute or drop.")


MISSING VALUES BY COLUMN
                  Missing  Missing %
OrderNumber             0        0.0
Sales Channel           0        0.0
WarehouseCode           0        0.0
ProcuredDate            0        0.0
OrderDate               0        0.0
ShipDate                0        0.0
DeliveryDate            0        0.0
CurrencyCode            0        0.0
_SalesTeamID            0        0.0
_CustomerID             0        0.0
_StoreID                0        0.0
_ProductID              0        0.0
Order Quantity          0        0.0
Discount Applied        0        0.0
Unit Cost               0        0.0
Unit Price              0        0.0

FINAL RESULT
Total missing cells:  0 of 127,856 (0.00%)
Columns affected:     0 of 16
Rows with any missing: 0 of 7,991
Verdict: Dataset is complete. No imputation or dropping needed.


In [4]:
print(df.dtypes)

OrderNumber          object
Sales Channel        object
WarehouseCode        object
ProcuredDate         object
OrderDate            object
ShipDate             object
DeliveryDate         object
CurrencyCode         object
_SalesTeamID          int64
_CustomerID           int64
_StoreID              int64
_ProductID            int64
Order Quantity        int64
Discount Applied    float64
Unit Cost            object
Unit Price           object
dtype: object


## 3. Data Type Conversion and Validation

This section converts date columns into datetime format and monetary columns into numeric format. Conversion results are validated by checking for parsing failures, ensuring that the transformed data is suitable for subsequent analysis while preserving the original dataset.

In [5]:
# Create a working copy; preserve the original DataFrame
clean_df = df.copy()

date_cols = ['ProcuredDate', 'OrderDate', 'ShipDate', 'DeliveryDate']
money_cols = ['Unit Cost', 'Unit Price']

# Convert dates (DD-MM-YYYY)
for col in date_cols:
    clean_df[col] = pd.to_datetime(df[col], format='%d-%m-%Y', errors='coerce')

# Convert money (strip thousands separators)
for col in money_cols:
    clean_df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

# Build the report: compare against the ORIGINAL data
rows = []
for col in date_cols + money_cols:
    originally_missing = int(df[col].isna().sum())
    missing_after = int(clean_df[col].isna().sum())
    rows.append({
        'Column': col,
        'Type': 'date' if col in date_cols else 'money',
        'Dtype before': str(df[col].dtype),
        'Dtype after': str(clean_df[col].dtype),
        'Originally missing': originally_missing,
        'Conversion failures': missing_after - originally_missing,
    })

report = pd.DataFrame(rows).set_index('Column')

print(f"\n{'=' * 75}\nCONVERSION REPORT\n{'=' * 75}")
print(report.to_string())

# Final result
total_failures = int(report['Conversion failures'].sum())
print(f"\n{'=' * 75}\nFINAL RESULT\n{'=' * 75}")
print(f"Columns converted:    {len(report)} ({len(date_cols)} date, {len(money_cols)} money)")
print(f"Conversion failures:  {total_failures:,}")

if total_failures == 0:
    print("Verdict: All values converted cleanly. Safe to proceed.")
else:
    bad = report[report['Conversion failures'] > 0].index.tolist()
    print(f"Verdict: Check these columns before continuing: {bad}")
    # Show examples of values that failed
    for col in bad:
        mask = clean_df[col].isna() & df[col].notna()
        print(f"\nSample failed values in '{col}':")
        print(df.loc[mask, col].head(5).to_string())


CONVERSION REPORT
               Type Dtype before     Dtype after  Originally missing  Conversion failures
Column                                                                                   
ProcuredDate   date       object  datetime64[ns]                   0                    0
OrderDate      date       object  datetime64[ns]                   0                    0
ShipDate       date       object  datetime64[ns]                   0                    0
DeliveryDate   date       object  datetime64[ns]                   0                    0
Unit Cost     money       object         float64                   0                    0
Unit Price    money       object         float64                   0                    0

FINAL RESULT
Columns converted:    6 (4 date, 2 money)
Conversion failures:  0
Verdict: All values converted cleanly. Safe to proceed.


## 4. Date Logic Validation

This section validates the chronological consistency of the dataset by checking the relationships between procurement, order, shipping, and delivery dates. The objective is to identify records with logically inconsistent date sequences before proceeding with further analysis.

In [7]:
# Validate chronological order of dates
checks = {
    'Procured after order':  clean_df['ProcuredDate'] > clean_df['OrderDate'],
    'Order after ship':      clean_df['OrderDate']    > clean_df['ShipDate'],
    'Ship after delivery':   clean_df['ShipDate']     > clean_df['DeliveryDate'],
    'Delivery before order': clean_df['DeliveryDate'] < clean_df['OrderDate'],
}

report = pd.DataFrame({
    'Violations': {name: int(mask.sum()) for name, mask in checks.items()},
})
report['% of rows'] = (report['Violations'] / len(clean_df) * 100).round(2)

print(f"\n{'=' * 45}\nDATE LOGIC VALIDATION\n{'=' * 45}")
print(report.to_string())

# Final result
any_violation = pd.concat(checks.values(), axis=1).any(axis=1)
print(f"\n{'=' * 45}\nFINAL RESULT\n{'=' * 45}")
print(f"Rows with any violation: {int(any_violation.sum()):,} of {len(clean_df):,}")
print("Verdict:", "All dates are in valid order." if not any_violation.any()
      else "Violations found. Inspect those rows before continuing.")


DATE LOGIC VALIDATION
                       Violations  % of rows
Procured after order            0        0.0
Order after ship                0        0.0
Ship after delivery             0        0.0
Delivery before order           0        0.0

FINAL RESULT
Rows with any violation: 0 of 7,991
Verdict: All dates are in valid order.


## 5. Numerical Data Validation

This section examines the numerical columns to identify unexpected or potentially invalid values in order quantities, discount rates, unit costs, and unit prices. The objective is to confirm that these fields fall within reasonable ranges and are suitable for subsequent sales calculations.

In [14]:
# Validate numerical columns
df_ = clean_df

checks = {
    'Order Quantity':   ('Order Quantity',   'Zero or negative',   df_['Order Quantity'] <= 0),
    'Discount Applied': ('Discount Applied', 'Outside [0, 1]',     (df_['Discount Applied'] < 0) | (df_['Discount Applied'] > 1)),
    'Unit Cost':        ('Unit Cost',        'Zero or negative',   df_['Unit Cost'] <= 0),
    'Unit Price':       ('Unit Price',       'Zero or negative',   df_['Unit Price'] <= 0),
}

rows = []
for name, (col, rule, mask) in checks.items():
    rows.append({
        'Column': name,
        'Min': df_[col].min(),
        'Max': df_[col].max(),
        'Rule': rule,
        'Violations': int(mask.sum()),
    })

report = pd.DataFrame(rows).set_index('Column')

# Cross-column check
price_below_cost = int((df_['Unit Price'] < df_['Unit Cost']).sum())

print(f"\n{'=' * 70}\nNUMERICAL DATA VALIDATION\n{'=' * 70}")
print(report.to_string(float_format=lambda x: f"{x:,.2f}"))
print(f"\nCross-check: Unit Price < Unit Cost -> {price_below_cost:,} rows")

# Final result
total_violations = int(report['Violations'].sum()) + price_below_cost
all_checks = [mask for _, (_, _, mask) in checks.items()]
all_checks.append(df_['Unit Price'] < df_['Unit Cost'])

any_violation = pd.concat(all_checks, axis=1).any(axis=1)

print(f"\n{'=' * 70}\nFINAL RESULT\n{'=' * 70}")
print(f"Total rule violations: {total_violations:,}")
print(f"Rows with any violation: {int(any_violation.sum()):,} of {len(df_):,}")


NUMERICAL DATA VALIDATION
                    Min      Max              Rule  Violations
Column                                                        
Order Quantity     1.00     8.00  Zero or negative           0
Discount Applied   0.05     0.40    Outside [0, 1]           0
Unit Cost         68.68 5,498.56  Zero or negative           0
Unit Price       167.50 6,566.00  Zero or negative           0

Cross-check: Unit Price < Unit Cost -> 0 rows

FINAL RESULT
Total rule violations: 0
Rows with any violation: 0 of 7,991


## 6. Categorical Data Validation

This section examines the categorical columns to identify distinct values, review their frequency distributions, and check for unexpected or inconsistent categories. The objective is to understand the dataset's categorical structure and confirm that these fields are suitable for grouping and comparative analysis.

In [17]:

# Validate categorical columns
categorical_cols = ['CurrencyCode', 'Sales Channel', 'WarehouseCode']

# 1. Summary table
rows = []

for col in categorical_cols:
    s = clean_df[col]
    non_missing = s.dropna()
    normalized = non_missing.astype(str).str.strip().str.lower()

    vc = s.value_counts(dropna=True)

    # Detect normalized label groups with multiple original spellings
    label_mapping = pd.DataFrame({
        'Original': non_missing.astype(str).values,
        'Normalized': normalized.values
    })

    label_issues = (
        label_mapping.groupby('Normalized')['Original']
        .nunique()
        .gt(1)
        .sum()
    )

    rows.append({
        'Column': col,
        'Unique': s.nunique(),
        'Missing': int(s.isna().sum()),
        'Top category': vc.index[0] if not vc.empty else 'N/A',
        'Top %': round(vc.iloc[0] / s.notna().sum() * 100, 1)
                 if s.notna().sum() > 0 else 0,
        'Label issues': int(label_issues)
    })

summary = pd.DataFrame(rows).set_index('Column')

print(f"\n{'=' * 70}\nCATEGORICAL DATA VALIDATION\n{'=' * 70}")
print(summary.to_string())

# 2. Frequency tables
for col in categorical_cols:
    vc = clean_df[col].value_counts(dropna=False)
    freq = pd.DataFrame({
        'Count': vc,
        '%': (vc / len(clean_df) * 100).round(1)
    })

    print(f"\n{'-' * 70}\n{col.upper()}\n{'-' * 70}")
    print(freq.to_string())

# 3. Show inconsistent labels, if any
print(f"\n{'=' * 70}\nLABEL INCONSISTENCY DETAILS\n{'=' * 70}")

for col in categorical_cols:
    s = clean_df[col].dropna().astype(str)
    label_mapping = pd.DataFrame({
        'Original': s,
        'Normalized': s.str.strip().str.lower()
    })

    inconsistent_groups = (
        label_mapping.groupby('Normalized')['Original']
        .nunique()
    )
    inconsistent_groups = inconsistent_groups[inconsistent_groups > 1]

    if inconsistent_groups.empty:
        print(f"{col}: No inconsistent labels detected.")
    else:
        print(f"\n{col}:")
        for normalized_label in inconsistent_groups.index:
            variants = sorted(
                label_mapping.loc[
                    label_mapping['Normalized'] == normalized_label,
                    'Original'
                ].unique()
            )
            print(f"  {normalized_label!r}: {variants}")

# 4. Final result
total_missing = int(summary['Missing'].sum())
total_issues = int(summary['Label issues'].sum())

print(f"\n{'=' * 70}\nFINAL RESULT\n{'=' * 70}")
print(f"Columns checked:       {len(categorical_cols)}")
print(f"Missing values:        {total_missing:,}")
print(f"Inconsistent groups:   {total_issues:,}")

print(
    "Verdict:",
    "Categorical columns passed these checks."
    if total_missing == 0 and total_issues == 0
    else "Potential issues found. Review the report before proceeding."
)



CATEGORICAL DATA VALIDATION
               Unique  Missing  Top category  Top %  Label issues
Column                                                           
CurrencyCode        1        0           USD  100.0             0
Sales Channel       4        0      In-Store   41.3             0
WarehouseCode       6        0  WARE-NMK1003   31.3             0

----------------------------------------------------------------------
CURRENCYCODE
----------------------------------------------------------------------
              Count      %
CurrencyCode              
USD            7991  100.0

----------------------------------------------------------------------
SALES CHANNEL
----------------------------------------------------------------------
               Count     %
Sales Channel             
In-Store        3298  41.3
Online          2425  30.3
Distributor     1375  17.2
Wholesale        893  11.2

----------------------------------------------------------------------
WAREHOUSECODE

## 7. Identifier Validation

This section validates the identifier columns used to link sales records with the product, sales team, customer, and store dimensions. The objective is to identify missing identifiers, unexpected values, and duplicate entries in the dimension lookup tables before preparing the data for relational analysis.

In [21]:
# Load dimension lookup tables from the processed Excel workbook
excel_path = '../data/processed/US_regional_sales_Analysis.xlsx'

sheets = ['DimProduct', 'DimSalesTeam', 'DimCustomer', 'DimStore']

# Passing a list of sheet names loads them all in one pass (returns a dict)
dimensions = pd.read_excel(excel_path, sheet_name=sheets)

dim_product    = dimensions['DimProduct']
dim_sales_team = dimensions['DimSalesTeam']
dim_customer   = dimensions['DimCustomer']
dim_store      = dimensions['DimStore']

# Summary table
rows = []
for name, dim in dimensions.items():
    rows.append({
        'Table': name,
        'Rows': len(dim),
        'Columns': dim.shape[1],
        'Duplicate rows': int(dim.duplicated().sum()),
        'Missing cells': int(dim.isna().sum().sum()),
        'Column names': ', '.join(dim.columns.astype(str)),
    })

summary = pd.DataFrame(rows).set_index('Table')

print(f"\n{'=' * 80}\nDIMENSION TABLES\n{'=' * 80}")
print(summary.drop(columns='Column names').to_string())

print(f"\n{'-' * 80}\nCOLUMNS\n{'-' * 80}")
for name, cols in summary['Column names'].items():
    print(f"{name:<14}: {cols}")

# Final result
total_dupes = int(summary['Duplicate rows'].sum())
total_missing = int(summary['Missing cells'].sum())
print(f"\n{'=' * 80}\nFINAL RESULT\n{'=' * 80}")
print(f"Tables loaded:    {len(dimensions)} of {len(sheets)}")
print(f"Duplicate rows:   {total_dupes:,}")
print(f"Missing cells:    {total_missing:,}")
print("Verdict:", "All dimension tables loaded cleanly." if total_dupes + total_missing == 0
      else "Issues found. Review the affected tables before joining.")


DIMENSION TABLES
              Rows  Columns  Duplicate rows  Missing cells
Table                                                     
DimProduct      47        1               0              0
DimSalesTeam    28        1               0              0
DimCustomer     50        1               0              0
DimStore       367        1               0              0

--------------------------------------------------------------------------------
COLUMNS
--------------------------------------------------------------------------------
DimProduct    : ProductID
DimSalesTeam  : SalesTeamID
DimCustomer   : CustomerID
DimStore      : StoreID

FINAL RESULT
Tables loaded:    4 of 4
Duplicate rows:   0
Missing cells:    0
Verdict: All dimension tables loaded cleanly.


In [25]:

# Validate identifier relationships between sales data and dimensions

id_checks = {
    '_ProductID': ('DimProduct', 'ProductID'),
    '_SalesTeamID': ('DimSalesTeam', 'SalesTeamID'),
    '_CustomerID': ('DimCustomer', 'CustomerID'),
    '_StoreID': ('DimStore', 'StoreID'),
}

rows = []

for fact_col, (dim_name, dim_col) in id_checks.items():
    fact_ids = clean_df[fact_col]
    dimension_ids = dimensions[dim_name][dim_col]

    unmatched_mask = ~fact_ids.isin(dimension_ids)

    rows.append({
        'Sales ID column': fact_col,
        'Dimension table': dim_name,
        'Unique sales IDs': fact_ids.nunique(),
        'Unique dimension IDs': dimension_ids.nunique(),
        'Missing sales IDs': int(fact_ids.isna().sum()),
        'Unmatched sales rows': int(unmatched_mask.sum()),
        'Duplicate dimension IDs': int(dimension_ids.duplicated().sum()),
    })

id_report = pd.DataFrame(rows).set_index('Sales ID column')

print(f"\n{'=' * 100}\nIDENTIFIER RELATIONSHIP VALIDATION\n{'=' * 100}")
print(id_report.to_string())

# Overall result
check_cols = [
    'Missing sales IDs',
    'Unmatched sales rows',
    'Duplicate dimension IDs',
]

total_issues = int(id_report[check_cols].to_numpy().sum())

print(f"\n{'=' * 100}\nFINAL RESULT\n{'=' * 100}")
print(f"Relationships checked: {len(id_checks)}")
print(f"Total validation issues: {total_issues:,}")
print(
    "Verdict:",
    "All identifier relationships passed."
    if total_issues == 0
    else "Issues found. Inspect unmatched or duplicate IDs before proceeding."
)



IDENTIFIER RELATIONSHIP VALIDATION
                Dimension table  Unique sales IDs  Unique dimension IDs  Missing sales IDs  Unmatched sales rows  Duplicate dimension IDs
Sales ID column                                                                                                                          
_ProductID           DimProduct                47                    47                  0                     0                        0
_SalesTeamID       DimSalesTeam                28                    28                  0                     0                        0
_CustomerID         DimCustomer                50                    50                  0                     0                        0
_StoreID               DimStore               367                   367                  0                     0                        0

FINAL RESULT
Relationships checked: 4
Total validation issues: 0
Verdict: All identifier relationships passed.


## 8. Business Metric Preparation

This section prepares the validated dataset for business analysis by defining the calculations required to derive sales revenue, discounts, total costs, and profit. The objective is to establish consistent metric definitions before aggregating and interpreting sales performance across channels, warehouses, products, customers, and sales teams.

In [29]:
# Create business metrics using documented working assumptions
clean_df['Gross Sales']     = clean_df['Order Quantity'] * clean_df['Unit Price']
clean_df['Discount Amount'] = clean_df['Gross Sales'] * clean_df['Discount Applied']
clean_df['Net Sales']       = clean_df['Gross Sales'] - clean_df['Discount Amount']
clean_df['Total Cost']      = clean_df['Order Quantity'] * clean_df['Unit Cost']
clean_df['Profit']          = clean_df['Net Sales'] - clean_df['Total Cost']

metric_cols = ['Order Quantity', 'Unit Price', 'Discount Applied', 'Gross Sales',
               'Discount Amount', 'Net Sales', 'Total Cost', 'Profit']

# 1. Preview
print(f"\n{'=' * 90}\nBUSINESS METRIC PREVIEW (first 10 rows)\n{'=' * 90}")
print(clean_df[metric_cols].head(10).to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

# 2. Totals
money_cols = ['Gross Sales', 'Discount Amount', 'Net Sales', 'Total Cost', 'Profit']
totals = clean_df[money_cols].sum().to_frame('Total')

print(f"\n{'=' * 90}\nTOTALS\n{'=' * 90}")
print(totals.to_string(float_format=lambda x: f"{x:,.2f}"))

# 3. Integrity checks
checks = pd.Series({
    'Missing metric values':         int(clean_df[money_cols].isna().any(axis=1).sum()),
    'Net Sales > Gross Sales':       int((clean_df['Net Sales'] > clean_df['Gross Sales']).sum()),
    'Net Sales != Gross - Discount': int((~((clean_df['Gross Sales'] - clean_df['Discount Amount'])
                                           .sub(clean_df['Net Sales']).abs() < 1e-6)).sum()),
    'Loss-making rows (Profit < 0)': int((clean_df['Profit'] < 0).sum()),
}, name='Rows')

print(f"\n{'=' * 90}\nINTEGRITY CHECKS\n{'=' * 90}")
print(checks.to_frame().to_string())

# Final result
net = clean_df['Net Sales'].sum()
profit = clean_df['Profit'].sum()
errors = int(checks.iloc[:3].sum())

print(f"\n{'=' * 90}\nFINAL RESULT\n{'=' * 90}")
print(f"Net Sales:       {net:,.2f}")
print(f"Total Profit:    {profit:,.2f}")
print(f"Profit Margin:   {profit / net * 100:.2f}%")
print(f"Avg Discount:    {clean_df['Discount Amount'].sum() / clean_df['Gross Sales'].sum() * 100:.2f}% of gross sales")
print("Verdict:", "Metrics calculated correctly." if errors == 0
      else "Calculation issues found. Inspect those rows before continuing.")

# Additional integrity checks
additional_checks = pd.Series({
    'Negative discount amounts': int(
        (clean_df['Discount Amount'] < 0).sum()
    ),
    'Negative total costs': int(
        (clean_df['Total Cost'] < 0).sum()
    ),
})

print("\n" + "=" * 90)
print("ADDITIONAL INTEGRITY CHECKS")
print("=" * 90)
print(additional_checks.to_frame().to_string())


BUSINESS METRIC PREVIEW (first 10 rows)
 Order Quantity  Unit Price  Discount Applied  Gross Sales  Discount Amount  Net Sales  Total Cost   Profit
              5    1,963.10              0.07     9,815.50           736.16   9,079.34    5,005.90 4,073.44
              3    3,939.60              0.07    11,818.80           886.41  10,932.39   10,045.98   886.41
              1    1,775.50              0.05     1,775.50            88.78   1,686.72      781.22   905.50
              8    2,324.90              0.07    18,599.20         1,394.94  17,204.26   11,717.52 5,486.74
              8    1,822.40              0.10    14,579.20         1,457.92  13,121.28   11,809.12 1,312.16
              5    1,038.50              0.05     5,192.50           259.62   4,932.88    2,232.80 2,700.07
              4    1,192.60              0.15     4,770.40           715.56   4,054.84    2,146.68 1,908.16
              5    1,815.70              0.05     9,078.50           453.93   8,624.58    7,625